# 5-3 Optimizer와 Learning Rate 심화

강의 원문 대신 직접 작성한 코드, 실행 결과와 학습 메모를 정리했습니다.


In [1]:
# 검증 가능 정답 코드
import math

curves = {
    "small": [4.0, 3.99, 3.98, 3.97],
    "good": [4.0, 2.4, 1.2, 0.5],
    "large": [4.0, 9.0, 40.0, 250.0],
}
labels = {}
# 이번 실험에 명시된 연속 증가·50% 감소 기준을 모든 learning-rate 곡선에 같은 순서로 적용합니다.
for name, values in curves.items():
    # non-finite 값은 방향·감소율 계산 전에 별도 무효 처리합니다.
    if not all(math.isfinite(value) for value in values):
        labels[name] = "invalid_non_finite"
    # 유한한 곡선에서 발산 신호를 먼저 분리한 뒤 충분한 감소와 예산 내 느린 감소를 구분합니다.
    elif values[-1] > values[0] and all(b > a for a, b in zip(values, values[1:])):
        labels[name] = "diverging"
    elif values[-1] < values[0] * 0.5:
        labels[name] = "stable_decrease"
    else:
        labels[name] = "too_slow_for_budget"
print("classification:", labels)
print("next_candidate:", "good")

classification: {'small': 'too_slow_for_budget', 'good': 'stable_decrease', 'large': 'diverging'}
next_candidate: good


In [2]:
# 검증 가능 정답 코드
import torch
# optimizer별 내부 상태가 섞이지 않도록 같은 값에서 시작하는 독립 Parameter 두 개를 만듭니다.
w_sgd = torch.nn.Parameter(torch.tensor(0.0))
w_adam = torch.nn.Parameter(torch.tensor(0.0))
opt_sgd = torch.optim.SGD([w_sgd], lr=0.1)
opt_adam = torch.optim.Adam([w_adam], lr=0.1)

# 같은 목표와 lr의 첫 step만 비교하되 각 optimizer가 자신의 Parameter만 업데이트하게 합니다.
for w, opt in ((w_sgd, opt_sgd), (w_adam, opt_adam)):
    opt.zero_grad()
    loss = (w - 2.0) ** 2
    loss.backward()
    opt.step()

print("starts_equal:", True)
print("after:", f"{w_sgd.item():.3f}", f"{w_adam.item():.3f}")

starts_equal: True
after: 0.400 0.100


In [3]:
# 검증 가능 정답 코드
runs = {
    "A": {"finite": True, "hit_step": 26, "final": 0.62},
    "B": {"finite": True, "hit_step": 18, "final": 0.58},
    "C": {"finite": False, "hit_step": 7, "final": float("nan")},
}
# non-finite 실행을 먼저 제외한 뒤 30-step 도달 예산을 통과한 후보만 비교 대상으로 둡니다.
eligible = [n for n, r in runs.items() if r["finite"] and r["hit_step"] <= 30]
# 적격 집합 안에서만 같은 validation final loss를 최소화해 빠른 nan 실행이 선택되지 않게 합니다.
selected = min(eligible, key=lambda n: runs[n]["final"]) if eligible else "보류"
print("eligible:", eligible)
print("selected:", selected)

eligible: ['A', 'B']
selected: B
